In [119]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [120]:
import os

base_path = "Daten"

all_measurements = []  
measurement_id = 0      

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    
    # Nur Unterordner nehmen
    if os.path.isdir(folder_path):
        
        try:
            # CSVs laden
            acceleration = pd.read_csv(os.path.join(folder_path, "Accelerometer.csv"))
            gravity = pd.read_csv(os.path.join(folder_path, "Gravity.csv"))
            gyroscope = pd.read_csv(os.path.join(folder_path, "Gyroscope.csv"))
            tag = pd.read_csv(os.path.join(folder_path, "Tags.csv"))
            
            
            acceleration["Sensor"] = "Accelerometer"
            gravity["Sensor"] = "Gravity"
            gyroscope["Sensor"] = "Gyroscope"
            
            data = pd.concat([acceleration, gravity, gyroscope], ignore_index=True)
            
            # Zeit konvertieren 
            data["time"] = pd.to_datetime(data["time"])
            
            # Tag setzen
            data["Tag"] = tag["tag"].iloc[0]
            
            # Messungs-ID setzen
            data["ID"] = measurement_id
            
            measurement_id += 1
            
            # Speichern
            all_measurements.append(data)
        
        except Exception as e:
            print(f"Fehler in Ordner {folder}: {e}")

# Alle Messungen zusammenführen
final_df = pd.concat(all_measurements, ignore_index=True)


Trimmen der ersten und letzten 3 Sekunden

In [121]:
def trim_measurement(group, trim_seconds=3):
    min_time = group["seconds_elapsed"].min()
    max_time = group["seconds_elapsed"].max()
    return group.loc[
        (group["seconds_elapsed"] >= min_time + trim_seconds) &
        (group["seconds_elapsed"] <= max_time - trim_seconds)
    ]

# ID als separate Spalte sichern bevor groupby sie verschluckt
final_df["ID_backup"] = final_df["ID"]

df = (
    final_df
    .groupby("ID_backup", group_keys=False)
    .apply(trim_measurement)
    .reset_index(drop=True)
)


In [122]:
df.head()

,time,seconds_elapsed,z,y,x,Sensor,Tag,ID
0,2026-03-03 10:42:50.960621300,3.259621,0.650506,2.949518,1.446072,Accelerometer,Treppe runter,0
1,2026-03-03 10:42:50.970531000,3.269531,0.272985,2.613810,0.252541,Accelerometer,Treppe runter,0
2,2026-03-03 10:42:50.980440800,3.279441,0.918067,0.648394,-1.411997,Accelerometer,Treppe runter,0
3,2026-03-03 10:42:50.990350600,3.289351,1.931937,-3.180973,-4.285746,Accelerometer,Treppe runter,0
4,2026-03-03 10:42:51.000260600,3.299260,1.467552,-2.795876,-4.884227,Accelerometer,Treppe runter,0


In [123]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 235629 entries, 0 to 235628
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   time             235629 non-null  datetime64[ns]
 1   seconds_elapsed  235629 non-null  float64       
 2   z                235629 non-null  float64       
 3   y                235629 non-null  float64       
 4   x                235629 non-null  float64       
 5   Sensor           235629 non-null  str           
 6   Tag              235629 non-null  str           
 7   ID               235629 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(1), str(2)
memory usage: 14.4 MB


In [124]:
df["Sensor"].value_counts()

Sensor
Accelerometer    89436
Gravity          89435
Gyroscope        56758
Name: count, dtype: int64

Wieso gibt es weniger Gyroscope Daten?

In [125]:
# IDs die Gyroscope-Einträge haben
ids_with_gyro = df[df['Sensor'] == 'Gyroscope']['ID'].unique()

# IDs die KEINEN Gyroscope-Eintrag haben
ids_without_gyro = df[~df['ID'].isin(ids_with_gyro)]['ID'].unique()

print(ids_without_gyro)

[]


In [126]:
df.groupby(['ID', 'Sensor']).size().unstack(fill_value=0)

Sensor,Accelerometer,Gravity,Gyroscope
ID,,,
0,2681,2682,2684
1,1232,1233,1235
2,5342,5342,5342
3,1159,1159,669
4,1266,1267,731
5,2239,2239,1290
6,1441,1441,830
7,3071,3070,1770
8,1887,1887,1087


In [127]:
df.groupby(['ID', 'Sensor'])['seconds_elapsed'].apply(
    lambda x: x.iloc[1] - x.iloc[0]
).unstack(fill_value=0)

Sensor,Accelerometer,Gravity,Gyroscope
ID,,,
0,0.009910,0.009910,0.009910
1,0.009910,0.009910,0.009910
2,0.009910,0.009910,0.009910
3,0.010000,0.010000,0.017346
4,0.010000,0.010000,0.017346
5,0.010000,0.010000,0.017346
6,0.010000,0.010000,0.017346
7,0.010000,0.010000,0.017347
8,0.010000,0.010000,0.017347


Das Problem ist, dass es nicht die gleichen Frequenzen aufnimmt. Die ersten 3 Spalten sind mit einem anderen Smartphone aufgenommen worden wie die anderen. In den Einstellungen der App wurde jedoch immer 100Hz angegeben.

Das Problem liegt wahrscheinlich daran, dass das Handy die Daten in Batches speichert und nicht genau in Echtzeit. Dewegen funktionieren Frequenzen, die im Verhältnis zu 60Hz stehen besser als andere. 

24 --> 60Hz

25 --> 80Hz

26 --> 120Hz